# 07. 기준표에서 모델 입력 꺼내기

06에서 만든 `cal01_base_v1.csv`를 읽음. 라벨과 파생컬럼을 다시 만들지 않고 RF 입력 후보 X, 임시 LABEL y, 식별 정보를 나눔. 모델 학습은 안 함.

In [ ]:
import os
import json
import pandas as pd
from IPython.display import display

if os.path.isfile(os.path.join('data', 'README.md')):
    ROOT = os.getcwd()
elif os.path.isfile(os.path.join('..', 'data', 'README.md')):
    ROOT = os.path.abspath('..')
else:
    ROOT = os.path.abspath(os.path.join('..', '..'))

ENC = 'utf-8-sig'
BASE_VERSION = 'v1'
DATA_DIR = os.path.join(ROOT, 'data', 'processed')
BASE_PATH = os.path.join(DATA_DIR, 'cal01_base_' + BASE_VERSION + '.csv')
INPUT_META = os.path.join(DATA_DIR, 'classification', BASE_VERSION, 'dataset.json')

# 저장한 숫자의 소수점 끝자리까지 같은 값으로 읽음.
master = pd.read_csv(
    BASE_PATH, encoding=ENC, parse_dates=['MEAS_DT', 'FEATURE_MINUTE'],
    dtype={'LABEL': 'Int8'}, float_precision='round_trip', low_memory=False,
)

# 기준표를 만들 때의 강종 원핫 자료형을 유지함.
for column in master.columns:
    if column.startswith('GRADE_'):
        master[column] = master[column].astype('int8')

with open(INPUT_META, 'r', encoding='utf-8') as file:
    contract = json.load(file)


## 기준표의 행 구분

0은 이번 규칙 미검출, 1은 신호와 정비 근거를 함께 만족한 임시 라벨임. 사용자 제외 행의 LABEL은 비어 있음.

In [ ]:
print('전체 기준표:', master.shape)
print('사용자 제외:', int(master['EXCLUDED'].sum()))

display(master['LABEL'].value_counts(dropna=False).rename('행 수'))


## 컬럼은 이렇게 보면 됨

전체 106개 중 RF 입력 후보 67개, 목표값 LABEL 1개, 식별, 검토용 38개임. CSV 전체를 X에 넣는 것은 아님.

| 컬럼, 묶음 | 의미와 사용 |
| --- | --- |
| ROW_ID / LOT_NO / MEAS_DT / GRADE | 행, LOT, 시각, 강종 식별. 분할, 결과 확인용 |
| CLN-IN / TC-* / OP-* / CP-Z4 | 세정 입측 온도, 각 존 온도, 제어출력, CP 기록값. 원본 21계측 입력 후보 |
| CP-Z4M | 해석, 생성 관계 확인이 남은 보조계측. 원본만 보존하고 X에서 제외 |
| LOT_ELAPSED_MIN / FEATURE_MINUTE | 원본 LOT 첫 관측 후 경과 분 / 과거 특징을 붙일 기준 분. 전자는 기존 67개 입력 후보에 포함, 후자는 조인용 |
| GRADE_* | 해당 강종이면 1, 아니면 0인 7개 컬럼. 정상, 이상 표시가 아님 |
| 센서_1분전 / _5분전 | 1분 전 / 5분 전의 분 평균 |
| 센서_직전10분평균 / _직전10분표준편차 | 직전 10개 분 평균의 평균 / 표준편차. 현재 분은 제외 |
| 센서_직전두분차이 | 직전 분 평균 - 2분 전 평균. 시간으로 나눈 변화율은 아님 |
| OP-Z3/Z4/Z5_직전10분포화율 | 직전 10분의 유효 원본 관측 중 출력 100 이상 비율(%) |
| EXCLUDED / EXCLUSION_REASON | 사용자 지정 제외 여부 / 이유. 제외 행의 LABEL은 비움 |
| FEATURES_COMPLETE / MODEL_ELIGIBLE | 입력 67개가 모두 유한한 숫자인지 / 제외되지 않고 입력과 LABEL이 있는지. 학습 완료 표시는 아님 |
| LABEL / LABEL_STATUS / LABEL_VERSION | 임시 0, 1 / 판정 상태 / 규칙 버전. LABEL만 y이며 X에는 넣지 않음 |
| LABEL_REASON / RULE / EVIDENCE_CONFIDENCE | 판정 이유, 규칙, 근거 한계 설명. 확률 점수가 아니며 X 제외 |
| EVIDENCE_WO / MNT_TYPE / SENSOR / CANDIDATE_IDS | 실제 컬럼은 모두 EVIDENCE_ 접두어임. 1 판정에 사용한 정비번호, 정비유형, 센서, 후보 ID |
| EVIDENCE_DOWN_MIN / HAS_PART_RECORD / PART_NAMES / PART_QTY_TOTAL / PART_CHANGE_REASONS / FAIL_MODE | 모두 EVIDENCE_ 접두어임. 근거 정비의 정지 분량, 교체기록 유무, 부품명, 수량, 사유, 고장모드. WO:값 형식이며 X 제외 |
| ASSOCIATION_KIND / EVIDENCE_INPUT_ALLOWED | 감지 전24시간, 작업 종료 후24시간의 연결 유형 / 근거의 X 사용 금지(False) 표시 |
| ANY_SENSOR_CANDIDATE / 각 *_CONTEXT / MAINT_WORK_OVERLAP / GRADE_CHANGE_NEARBY_30MIN | 후보, 제어반응, 주기, LOT 경계, 수리, 작업, 강종변경 맥락. 모두 검토용이며 이상 확정이나 실제 정지 표시가 아님 |
| LOT_CONTEXT_ID / SENSOR_CANDIDATE_IDS | LOT의 사건 근거 / 겹치는 전체 센서 후보를 찾아가는 ID. 채택 후보만 담은 EVIDENCE_CANDIDATE_IDS와 구분 |

과거 5종은 TC/OP-Z3, Z4, Z5와 CP-Z4의 7개 센서에 붙어 35개, OP 포화율 3개를 더해 총 38개임. 근거 문자열은 인코딩해도 X에서 제외함. DOWN_MIN은 정지 분량이지 해당 행 시각의 정지 여부가 아님.

## RF 입력 후보 X와 임시 정답 y

기존 RF 후보 67개가 모두 있고 사용자 제외가 아닌 행을 선택함. 텍스트 근거와 LABEL은 X에 넣지 않음. 이 필터는 RF 후보 기준이며 Z-score는 08에서 따로 준비함.

In [ ]:
input_columns = contract['method_input_columns']['RandomForestClassifier']

selected = master.loc[master['MODEL_ELIGIBLE'] == True].copy()

X = selected[input_columns].copy()

y = selected['LABEL'].astype('int8')

row_keys = selected[['ROW_ID', 'LOT_NO', 'MEAS_DT', 'GRADE']].copy()

print('RF 입력 후보 X:', X.shape)
print('임시 정답 y:', y.shape)

display(y.value_counts().rename('행 수'))
display(X.head())


## 다음 단계

Z-score 입력은 08에서 확인함. IF의 입력 컬럼과 회귀의 목표 센서, 미래 시차는 아직 미정임. 시간과 LOT 분할, 9:1 추출, 모델 실행은 여기서 하지 않음.